<a href="https://colab.research.google.com/github/lahari809-rgb/Machine-Learning/blob/main/Experiment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload placement_predict_50k_Dataset.csv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)

pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('placement_predict_50k Dataset (1).csv')

print(df.shape)

df.head()

print(df.isnull().sum()[df.isnull().sum() > 0])

print(df['PlacementStatus'].value_counts())
print(df['CGPA_Tier'].value_counts())

In [ ]:
drop_common = ['StudentID', 'IsAnomaly', 'Salary Package']

categorical_cols = [
    'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation',
    'Hostel', 'HistoryOfBacklogs', 'ExtraCurricular'
]

numeric_cols = [
    'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5',
    'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent',
    'Internships', 'Projects', 'Workshops', 'Certifications',
    'Publications', 'AptitudeTestScore', 'SoftSkillsRating',
    'CodingTestScore', 'MockInterviewScore'
]

print(len(categorical_cols), 'categorical |', len(numeric_cols), 'numeric')

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

In [ ]:
def evaluate_classifier(model, X_test, y_test, class_names=None, average='binary'):
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average=average, zero_division=0)
    rec = recall_score(y_test, y_pred, average=average, zero_division=0)
    f1 = f1_score(y_test, y_pred, average=average, zero_division=0)

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    if average == 'binary' and hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_proba)
        print(f"ROC-AUC  : {auc:.4f}")

        fpr, tpr, _ = roc_curve(y_test, y_proba)
        plt.figure(figsize=(5, 5))
        plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
        plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC Curve')
        plt.legend()
        plt.show()

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, cmap='Blues')
    plt.title('Confusion Matrix')
    plt.colorbar()
    ticks = class_names if class_names is not None else np.unique(y_test)
    plt.xticks(range(len(ticks)), ticks, rotation=45)
    plt.yticks(range(len(ticks)), ticks)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha='center', va='center')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    print("\nClassification Report:\n", classification_report(y_test, y_pred, zero_division=0))

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

In [ ]:
X1 = df[numeric_cols + categorical_cols]
y1 = df['PlacementStatus']

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42, stratify=y1
)

placement_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

placement_model.fit(X1_train, y1_train)

In [ ]:
results_placement = evaluate_classifier(
    placement_model, X1_test, y1_test,
    class_names=['Not Placed', 'Placed'], average='binary'
)

In [ ]:
def get_feature_names(preprocessor, numeric_cols, categorical_cols):
    ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
    cat_names = ohe.get_feature_names_out(categorical_cols)
    return list(numeric_cols) + list(cat_names)

feature_names_1 = get_feature_names(
    placement_model.named_steps['preprocessor'], numeric_cols, categorical_cols
)

coefs_1 = placement_model.named_steps['classifier'].coef_[0]

coef_df_1 = pd.DataFrame({'feature': feature_names_1, 'coefficient': coefs_1})
coef_df_1 = coef_df_1.reindex(
    coef_df_1.coefficient.abs().sort_values(ascending=False).index
)

plt.figure(figsize=(9, max(6, len(coef_df_1) * 0.25)))
colors = ['#2E7D32' if c > 0 else '#C62828' for c in coef_df_1['coefficient']]
plt.barh(coef_df_1['feature'], coef_df_1['coefficient'], color=colors)
plt.gca().invert_yaxis()
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient (log-odds impact)')
plt.title('PlacementStatus Logistic Regression — Coefficients')
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols_tier = [c for c in numeric_cols if c != 'CGPA']

X2 = df[numeric_cols_tier + categorical_cols]
y2 = df['CGPA_Tier']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

preprocessor_tier = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols_tier),
    ('cat', categorical_transformer, categorical_cols)
])

tier_model = Pipeline(steps=[
    ('preprocessor', preprocessor_tier),
    ('classifier', LogisticRegression(
        max_iter=2000, multi_class='multinomial', solver='lbfgs', random_state=42
    ))
])

tier_model.fit(X2_train, y2_train)

In [ ]:
class_order = sorted(y2.unique())  # e.g. ['High','Low','Mid']

results_tier = evaluate_classifier(
    tier_model, X2_test, y2_test,
    class_names=class_order, average='macro'
)

In [ ]:
feature_names_2 = get_feature_names(
    tier_model.named_steps['preprocessor'], numeric_cols_tier, categorical_cols
)

classes_2 = tier_model.named_steps['classifier'].classes_
coefs_2 = tier_model.named_steps['classifier'].coef_

fig, axes = plt.subplots(len(classes_2), 1, figsize=(9, 5 * len(classes_2)))
if len(classes_2) == 1:
    axes = [axes]

for ax, cls, coef_row in zip(axes, classes_2, coefs_2):
    cdf = pd.DataFrame({'feature': feature_names_2, 'coefficient': coef_row})
    cdf = cdf.reindex(
        cdf.coefficient.abs().sort_values(ascending=False).index
    ).head(15)

    colors = ['#2E7D32' if c > 0 else '#C62828' for c in cdf['coefficient']]
    ax.barh(cdf['feature'], cdf['coefficient'], color=colors)
    ax.invert_yaxis()
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'CGPA_Tier Logistic Regression — Class: {cls} (top 15)')
    ax.set_xlabel('Coefficient (log-odds impact)')

plt.tight_layout()
plt.show()

In [ ]:
print("=== Model comparison ===")
print("PlacementStatus (binary):", results_placement)
print("CGPA_Tier (multinomial):", results_tier)